<a href="https://colab.research.google.com/github/hamshini1413/hamshini_gen_ai_foundation/blob/main/12_Knowledge_Distillation_of_Dual_Encoder_Vision_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision transformers scipy

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import CLIPProcessor
from transformers import CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

###################################################
# Teacher Model
###################################################

teacher = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

teacher.eval()

###################################################
# Student Model
###################################################

class StudentModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(512,256),
            nn.ReLU(),

            nn.Linear(256,512)

        )

    def forward(self,x):

        return self.encoder(x)

student = StudentModel().to(device)

###################################################
# Dummy Data
###################################################

batch = 16

teacher_features = torch.randn(batch,512).to(device)

###################################################
# Forward
###################################################

student_features = student(teacher_features)

###################################################
# Cosine Loss
###################################################

cosine_loss = 1 - F.cosine_similarity(

    teacher_features,

    student_features

).mean()

###################################################
# KL Loss
###################################################

teacher_logits = F.log_softmax(

    teacher_features,

    dim=1

)

student_logits = F.softmax(

    student_features,

    dim=1

)

kl_loss = nn.KLDivLoss(

    reduction="batchmean"

)(teacher_logits,student_logits)

###################################################
# Final Loss
###################################################

loss = 0.5*cosine_loss + 0.5*kl_loss

print(loss.item())

###################################################
# Training
###################################################

optimizer = torch.optim.Adam(

    student.parameters(),

    lr=1e-4

)

optimizer.zero_grad()

loss.backward()

optimizer.step()

print("Student Updated")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

0.7611438035964966
Student Updated
